# R7 — Exact "accept-all" attack-verdict counts for the asymmetric policy (all detectors, all datasets). POST HOC.
For every window: total flows, attack flows, correct attack verdicts (TP_all) and false alarms (FA_all) with **no threshold**.
Each part checks window alignment against the saved results (attack counts must match exactly) before saving.
Run H0 first, then any of H1–H4 (each is self-contained after H0), then H5. GPU runtime recommended.

### CELL H0 — common setup

In [ ]:
# CELL H0 — common setup
from google.colab import drive
drive.mount("/content/drive")
import os, gc, json, glob, shutil, time
import numpy as np, pandas as pd
BASE = "/content/drive/MyDrive/TrustShift_IDS"; V2 = f"{BASE}/V2_adaptive_reliability"
OUT = f"{V2}/revision_R7_acceptall"; os.makedirs(OUT, exist_ok=True)
W = 5000
def counts(pred, y):
    return {"n": int(len(y)), "n_attack": int((y == 1).sum()), "TP_all": int(((pred == 1) & (y == 1)).sum()),
            "FA_all": int(((pred == 1) & (y == 0)).sum())}
def gate(new, saved, keys, label):
    m = new.merge(saved[keys + ["n_attack"]].drop_duplicates(keys), on=keys, suffixes=("", "_saved"), how="left")
    bad = int((m["n_attack"] != m["n_attack_saved"]).sum())
    print(f"[gate] {label}: {len(m)} windows, attack-count mismatches = {bad}", "PASS" if bad == 0 else "FAIL")
    assert bad == 0, f"window alignment failed for {label}"
print("[H0] COMPLETE")

### CELL H1 — CICIDS2017 XGBoost: deployment sequences and CSE-CIC-IDS2018 (from stored predictions)

In [ ]:
# CELL H1 — CICIDS2017 XGBoost: deployment sequences and CSE-CIC-IDS2018 (from stored predictions)
import joblib
T_B = 0.9744155346869672
FEATURE_ORDER = list(json.load(open(f"{V2}/controlled_shifts/S3_corruption/V2_STEP_6D_S3_CORRUPTION_REALIZATION.json"))["feature_order"])
xgb_b = joblib.load(f"{BASE}/models/xgboost/XGB_binary_seed42.joblib")
def xgb_pred(X):
    p = np.clip(xgb_b.predict_proba(X), 1e-15, 1.0); z = np.log(p) / T_B; z -= z.max(1, keepdims=True)
    q = np.exp(z); q /= q.sum(1, keepdims=True); return q.argmax(1).astype(np.int64)
iid = pd.read_parquet(f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_iid_test_68F_imputed.parquet", columns=["binary_label"])
raw = iid["binary_label"]
y_iid = raw.to_numpy(np.int64) if np.issubdtype(raw.dtype, np.number) else raw.astype(str).str.lower().map({"benign":0,"attack":1,"0":0,"1":1}).to_numpy(np.int64)
rows = []
for seed in [1, 7, 21, 42, 84]:
    z = np.load(f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz", allow_pickle=False)
    pr = xgb_pred(np.asarray(z["X"], np.float64)); y = y_iid[np.asarray(z["source_row_index"], np.int64)]; z.close()
    for w in range(60): rows.append({"seed": seed, "window_id": w, **counts(pr[w*W:(w+1)*W], y[w*W:(w+1)*W])})
a = pd.DataFrame(rows); a.insert(0, "detector", "XGBoost"); a.insert(0, "dataset", "CIC_sequences")
sv = pd.read_csv(f"{V2}/revision_R1/R2_multiseed_window_threshold_outcomes.csv.gz"); sv = sv[(sv.task == "binary") & (sv.thr_name == "state_normal")]
gate(a, sv, ["seed", "window_id"], "CIC XGBoost sequences")
# external: stored binary predictions (no model needed)
shards = sorted(glob.glob(f"{BASE}/results/shift_evaluation/S1_external/predictions/Step_12_2B_S1_part_*.parquet"))
P = np.concatenate([pd.read_parquet(s, columns=["binary_pred"]).binary_pred.to_numpy(np.int64) for s in shards])
Y = np.concatenate([pd.read_parquet(s, columns=["binary_true"]).binary_true.to_numpy(np.int64) for s in shards])
b = pd.DataFrame([{"window_id": w, **counts(P[w*W:(w+1)*W], Y[w*W:(w+1)*W])} for w in range(len(Y) // W)])
b.insert(0, "detector", "XGBoost"); b.insert(0, "dataset", "CIC_external")
sv = pd.read_csv(f"{V2}/revision_R1/R7_S1_binary_security_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "state_normal"]
gate(b, sv, ["window_id"], "CIC XGBoost CSE-CIC-IDS2018")
pd.concat([a, b]).to_csv(f"{OUT}/H1_cic_xgboost_acceptall.csv", index=False); del P, Y; gc.collect(); print("[H1] COMPLETE")

### CELL H2 — CICIDS2017 neural detectors (MLP, FT-Transformer): sequences and CSE-CIC-IDS2018

In [ ]:
# CELL H2 — CICIDS2017 neural detectors (MLP, FT-Transformer): sequences and CSE-CIC-IDS2018
import tensorflow as tf, keras, pyarrow.parquet as pq
@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, n_features, token_dim, **kw): super().__init__(**kw); self.n_features = int(n_features); self.token_dim = int(token_dim)
    def build(self, s):
        self.feature_weight = self.add_weight(name="feature_weight", shape=(self.n_features, self.token_dim), initializer="glorot_uniform", trainable=True)
        self.feature_bias = self.add_weight(name="feature_bias", shape=(self.n_features, self.token_dim), initializer="zeros", trainable=True); super().build(s)
    def call(self, x): return tf.expand_dims(x, -1) * self.feature_weight + self.feature_bias
    def get_config(self): c = super().get_config(); c.update({"n_features": self.n_features, "token_dim": self.token_dim}); return c
@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class CLSToken(tf.keras.layers.Layer):
    def __init__(self, token_dim, **kw): super().__init__(**kw); self.token_dim = int(token_dim)
    def build(self, s): self.cls_token = self.add_weight(name="cls_token", shape=(1, 1, self.token_dim), initializer="zeros", trainable=True); super().build(s)
    def call(self, x): return tf.concat([tf.tile(self.cls_token, [tf.shape(x)[0], 1, 1]), x], axis=1)
    def get_config(self): c = super().get_config(); c.update({"token_dim": self.token_dim}); return c
CUSTOM = {"NumericalFeatureTokenizer": NumericalFeatureTokenizer, "CLSToken": CLSToken,
          "TrustShiftIDS>NumericalFeatureTokenizer": NumericalFeatureTokenizer, "TrustShiftIDS>CLSToken": CLSToken}
M = {"MLP": keras.models.load_model(f"{BASE}/models/mlp/MLP_binary_seed42.keras", compile=False, custom_objects=CUSTOM, safe_mode=False),
     "FTTransformer": keras.models.load_model(f"{BASE}/models/ft_transformer/FTTransformer_binary_seed42.keras", compile=False, custom_objects=CUSTOM, safe_mode=False)}
CALN = json.load(open(f"{V2}/revision_R4/N1_neural_calibration_and_thresholds.json"))
SPEC = pd.read_csv(f"{BASE}/metadata/TrustShift_IDS_FROZEN_NEURAL_PREPROCESSING_SPEC.csv"); FEATURES = SPEC.feature_id.tolist()
C = SPEC[SPEC.feature_group == "continuous"]; CI = np.array([FEATURES.index(f) for f in C.feature_id])
LO, HI, MU, SD = (C[c].to_numpy(float) for c in ["lower_bound", "upper_bound", "mean", "std"])
def ntf(X): X = np.array(X, np.float64, copy=True); X[:, CI] = (np.clip(X[:, CI], LO, HI) - MU) / SD; return X
def npred(name, Xn):
    p = M[name].predict(Xn.astype(np.float32), batch_size=65536, verbose=0).reshape(-1).astype(np.float64)
    T = CALN[name]["T"]; p = np.clip(p, 1e-12, 1 - 1e-12); q = 1 / (1 + np.exp(-(np.log(p) - np.log1p(-p)) / T)); return (q >= 0.5).astype(np.int64)
iid = pd.read_parquet(f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_iid_test_68F_imputed.parquet", columns=["binary_label"])
raw = iid["binary_label"]
y_iid = raw.to_numpy(np.int64) if np.issubdtype(raw.dtype, np.number) else raw.astype(str).str.lower().map({"benign":0,"attack":1,"0":0,"1":1}).to_numpy(np.int64)
rows = []
for seed in [1, 7, 21, 42, 84]:
    z = np.load(f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz", allow_pickle=False)
    Xn = ntf(z["X"]); y = y_iid[np.asarray(z["source_row_index"], np.int64)]; z.close()
    for name in M:
        pr = npred(name, Xn)
        for w in range(60): rows.append({"dataset": "CIC_sequences", "detector": name, "seed": seed, "window_id": w, **counts(pr[w*W:(w+1)*W], y[w*W:(w+1)*W])})
a = pd.DataFrame(rows)
sv = pd.read_csv(f"{V2}/revision_R4/N2_neural_stream_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "fixed"]
for name in M: gate(a[a.detector == name], sv[sv.detector == name], ["seed", "window_id"], f"CIC {name} sequences")
NAME = dict(zip(SPEC.feature_id, SPEC.feature_name)); EXT_COLS = [NAME[f] for f in FEATURES]
parts = sorted(glob.glob(f"{BASE}/data/processed/CSE_CIC_IDS2018_68F_imputed/part_*.parquet"))
shards = sorted(glob.glob(f"{BASE}/results/shift_evaluation/S1_external/predictions/Step_12_2B_S1_part_*.parquet"))
PR = {n: [] for n in M}; Y = []
for pp, sp in zip(parts, shards):
    Xn = ntf(pd.read_parquet(pp, columns=EXT_COLS).to_numpy(np.float64)); Y.append(pd.read_parquet(sp, columns=["binary_true"]).binary_true.to_numpy(np.int64))
    for n in M: PR[n].append(npred(n, Xn))
    del Xn; gc.collect()
Y = np.concatenate(Y); rows = []
for n in M:
    pr = np.concatenate(PR[n])
    for w in range(len(Y) // W): rows.append({"dataset": "CIC_external", "detector": n, "window_id": w, **counts(pr[w*W:(w+1)*W], Y[w*W:(w+1)*W])})
b = pd.DataFrame(rows)
sv = pd.read_csv(f"{V2}/revision_R4/N3_neural_external_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "fixed"]
for n in M: gate(b[b.detector == n], sv[sv.detector == n], ["window_id"], f"CIC {n} CSE-CIC-IDS2018")
pd.concat([a, b]).to_csv(f"{OUT}/H2_cic_neural_acceptall.csv", index=False); del PR, Y; gc.collect(); print("[H2] COMPLETE")

### CELL H3 — NetFlow detectors: NF-UNSW-NB15-v2 sequences (exact R5 reconstruction)

In [ ]:
# CELL H3 — NetFlow detectors: NF-UNSW-NB15-v2 sequences (exact R5 reconstruction)
import xgboost as xgb
R5 = f"{V2}/revision_R5_netflow"; R5P = json.load(open(f"{R5}/F0_PROTOCOL_FROZEN.json")); FEAT = R5P["features"]
CAL5 = json.load(open(f"{R5}/F1_calibration_thresholds.json")); IMPORTANT = CAL5["importance_order"]
xg = xgb.XGBClassifier(); xg.load_model(f"{R5}/F1_xgb_source.json")
mlp5 = keras.models.load_model(f"{R5}/F1_mlp_source.keras", compile=False); pp5 = np.load(f"{R5}/F1_mlp_prep.npz"); mu5, sd5, MEDIAN = pp5["mu"], pp5["sd"], pp5["median"]
F32 = np.finfo(np.float32).max
def pred5(det, X):
    if det == "XGBoost":
        X = np.where(np.isnan(X), 0.0, np.clip(X, -F32, F32)); p = xg.predict_proba(X.astype(np.float32))[:, 1].astype(np.float64)
    else:
        p = mlp5.predict(((np.log1p(np.clip(X, 0, None)) - mu5) / sd5).astype(np.float32), batch_size=65536, verbose=0).reshape(-1).astype(np.float64)
    T = CAL5[det]["T"]; p = np.clip(p, 1e-12, 1 - 1e-12); q = 1 / (1 + np.exp(-(np.log(p) - np.log1p(-p)) / T)); return (q >= 0.5).astype(np.int64)
SRC = f"{BASE}/data/raw/NetFlow_v2/NF-UNSW-NB15-v2/NF-UNSW-NB15-v2.csv"
df = pd.read_csv(SRC, usecols=FEAT + ["Label"]); X_all = df[FEAT].to_numpy(np.float64); y_all = df["Label"].to_numpy(np.int64); del df
h = pd.util.hash_pandas_object(pd.DataFrame(X_all), index=False).to_numpy(); g = pd.DataFrame({"h": h, "y": y_all})
mixed = g.groupby("h").y.nunique(); keep = ~g.h.isin(set(mixed[mixed > 1].index)).to_numpy()
X_all, y_all, h = X_all[keep], y_all[keep], h[keep]
uh = np.unique(h); perm = np.random.default_rng(42).permutation(len(uh))
cut = np.cumsum([int(f * len(uh)) for f in R5P["split"]["fractions"]])[:-1]; pg = np.empty(len(uh), np.int8)
for i, idx in enumerate(np.split(perm, cut)): pg[idx] = i
part = pg[np.searchsorted(uh, h)]; Xte, yte = X_all[part == 3], y_all[part == 3]
assert np.allclose(np.median(X_all[part == 0], axis=0), MEDIAN), "split reconstruction differs from R5"
fidx = {f: i for i, f in enumerate(FEAT)}
def mask(X, stage, rng):
    X = X.copy(); n = X.shape[0]
    def rnd(p):
        k = int(round(p * 38)); cols = np.argsort(rng.random((n, 38)), axis=1)[:, :k]; X[np.arange(n)[:, None], cols] = MEDIAN[cols]
    def imp(p):
        k = int(round(p * 38)); cols = [fidx[f] for f in IMPORTANT[:k]]; X[:, cols] = MEDIAN[cols]
    if stage == "mask05": rnd(0.05)
    elif stage == "mask10": rnd(0.10)
    elif stage == "mask20": rnd(0.20)
    elif stage == "combined10": rnd(0.10); imp(0.10)
    return X
stages = R5P["sequences"]["stages"]; need = len(stages) * 10 * W; rows = []
for seed in R5P["sequences"]["orderings"]:
    rng = np.random.default_rng(seed); order = rng.permutation(len(Xte))[:need]
    for si, st in enumerate(stages):
        idx = order[si * 10 * W:(si + 1) * 10 * W]; Xs = mask(Xte[idx], st, rng); ys = yte[idx]
        for det in ["XGBoost", "MLP"]:
            pr = pred5(det, Xs)
            for w in range(10): rows.append({"dataset": "NetFlow_sequences", "detector": det, "seed": seed, "window_id": si * 10 + w, **counts(pr[w*W:(w+1)*W], ys[w*W:(w+1)*W])})
a = pd.DataFrame(rows); sv = pd.read_csv(f"{R5}/F2_sequence_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "fixed"]
for det in ["XGBoost", "MLP"]: gate(a[a.detector == det], sv[sv.detector == det], ["seed", "window_id"], f"NetFlow {det} sequences")
a.to_csv(f"{OUT}/H3_netflow_sequences_acceptall.csv", index=False); del X_all; gc.collect(); print("[H3] COMPLETE")

### CELL H4 — NetFlow detectors: NF-CSE-CIC-IDS2018-v2 and NF-ToN-IoT-v2 (chunked, file order)

In [ ]:
# CELL H4 — NetFlow detectors: NF-CSE-CIC-IDS2018-v2 and NF-ToN-IoT-v2 (chunked, file order)
def chunked(path, label):
    rows = []; carry = None; wid = 0
    for ch in pd.read_csv(path, usecols=FEAT + ["Label"], chunksize=1_000_000):
        Xc = ch[FEAT].to_numpy(np.float64); yc = ch["Label"].to_numpy(np.int64)
        if carry is not None: Xc = np.vstack([carry[0], Xc]); yc = np.concatenate([carry[1], yc])
        nf = (len(yc) // W) * W; carry = (Xc[nf:], yc[nf:]); Xc, yc = Xc[:nf], yc[:nf]
        for det in ["XGBoost", "MLP"]:
            pr = pred5(det, Xc)
            for w in range(nf // W): rows.append({"dataset": label, "detector": det, "window_id": wid + w, **counts(pr[w*W:(w+1)*W], yc[w*W:(w+1)*W])})
        wid += nf // W
    return pd.DataFrame(rows)
e = chunked(f"{BASE}/data/raw/NetFlow_v2/NF-CSE-CIC-IDS2018-v2/data/NF-CSE-CIC-IDS2018-v2.csv", "NetFlow_external")
sv = pd.read_csv(f"{R5}/F3_external_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "fixed"]
for det in ["XGBoost", "MLP"]: gate(e[e.detector == det], sv[sv.detector == det], ["window_id"], f"NetFlow {det} NF-CSE-CIC-IDS2018-v2")
t = chunked(f"{BASE}/data/raw/NF-ToN-IoT-v2/NF-ToN-IoT-v2.csv", "ToN_confirmation")
sv = pd.read_csv(f"{V2}/revision_R6_confirmation/G1_ToN_window_outcomes.csv.gz"); sv = sv[sv.thr_name == "fixed"]
for det in ["XGBoost", "MLP"]: gate(t[t.detector == det], sv[sv.detector == det], ["window_id"], f"NetFlow {det} NF-ToN-IoT-v2")
pd.concat([e, t]).to_csv(f"{OUT}/H4_netflow_external_acceptall.csv", index=False); print("[H4] COMPLETE")

### CELL H5 — package

In [ ]:
# CELL H5 — package
for f in os.listdir(OUT):
    if f.endswith(".zip"): os.remove(os.path.join(OUT, f))
z = shutil.make_archive(f"{V2}/TrustShift_revision_R7_export", "zip", OUT)
print("Upload:", z, f"({os.path.getsize(z)/1e6:.1f} MB)"); print("Contents:", sorted(os.listdir(OUT)))